In [1]:
# CELL 1 — Install dependencies
import subprocess, sys
 
subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "transformers>=4.50.0", "accelerate>=0.30.0",
     "sentencepiece", "protobuf",
     "pandas>=2.0.0", "numpy>=1.24.0",
     "tqdm>=4.66.0",
     "--quiet"],
    check=True
)
print("✅ All dependencies installed.")

✅ All dependencies installed.


In [2]:
# CELL 2A — Base imports (pandas/numpy FIRST — torch must come later)
import os, re, json, gc
from collections import Counter
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("✅ Base imports done")

✅ Base imports done


In [3]:
# CELL 2B — Torch + device check (separate cell, runs AFTER 2A)
import torch
 
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ CUDA  →  {gpu_name}  ({gpu_mem:.1f} GB VRAM)")
    DEVICE      = 0
    TORCH_DTYPE = torch.float16
else:
    print("⚠️  No GPU — falling back to CPU.")
    DEVICE      = -1
    TORCH_DTYPE = torch.float32

✅ CUDA  →  NVIDIA GeForce RTX 4060 Laptop GPU  (8.6 GB VRAM)


In [4]:
# CELL 3 — Config
INPUT_CSV  = "input_datasets/testing_hindi_fp_privacy_filter 2.csv"
OUTPUT_CSV = "predictions/predictions_FP_hin.csv"
MODEL_ID   = "openai/privacy-filter"
BATCH_SIZE = 1          # MoE+Viterbi decoder does not support HF pipeline batching
AGGREGATION = "simple"
 
COL_RAW         = "rawText"
COL_MASKED_GT   = "maskedText"
COL_SNO         = "s_no"
COL_CATEGORY    = "category"
COL_DIFFICULTY  = "difficulty"
COL_ADVERSARIAL = "adversarial_type"
 
def mask_token(label: str) -> str:
    return f"[{label.upper()}]"
 
ALL_LABELS = [
    "account_number", "private_address", "private_email",
    "private_person", "private_phone", "private_url",
    "private_date", "secret",
]
 
LABEL_TO_COL = {
    "account_number" : "num_account_number",
    "private_address": "num_private_address",
    "private_date"   : "num_private_date",
    "private_email"  : "num_private_email",
    "private_person" : "num_private_person",
    "private_phone"  : "num_private_phone",
    "private_url"    : "num_private_url",
    "secret"         : "num_secret",
}

In [5]:
#FIXING BROKEN CSV

# import csv

# INPUT_CSV1 = "input_datasets/input_FP_hin1.csv"
# OUTPUT_CSV1 = "input_datasets/input_FP_hin.csv"

# EXPECTED_COLS = 16

# fixed_rows = []

# with open(INPUT_CSV1, "r", encoding="utf-8", newline="") as f:
#     reader = csv.reader(f)

#     header = next(reader)
#     fixed_rows.append(header)

#     for row_num, row in enumerate(reader, start=2):

#         # already correct
#         if len(row) == EXPECTED_COLS:
#             fixed_rows.append(row)
#             continue

#         print(f"Fixing row {row_num}: {len(row)} cols")

#         # keep first 15 columns fixed
#         fixed_part = row[:15]

#         # merge remaining columns into trick_reason
#         trick_reason = ",".join(row[15:]).strip()

#         fixed_row = fixed_part + [trick_reason]

#         fixed_rows.append(fixed_row)

# # write repaired csv
# with open(OUTPUT_CSV1, "w", encoding="utf-8", newline="") as f:
#     writer = csv.writer(
#         f,
#         quoting=csv.QUOTE_ALL
#     )

#     writer.writerows(fixed_rows)

# print("Fixed CSV saved to:", OUTPUT_CSV1)

In [6]:
# CELL 4 — Load CSV
df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
 
missing = [c for c in [COL_RAW, COL_MASKED_GT] if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Found: {list(df.columns)}")
 
texts         = df[COL_RAW].astype(str).tolist()
ground_truths = df[COL_MASKED_GT].astype(str).tolist()
 
print(f"✅ Loaded {len(df):,} rows from '{INPUT_CSV}'")
count_cols = [c for c in df.columns if c.startswith("num_") or c == "numberOfPII"]
for col in count_cols:
    print(f"   {col:<25}: total={df[col].sum():,}  mean/row={df[col].mean():.2f}")

✅ Loaded 700 rows from 'input_datasets/testing_hindi_fp_privacy_filter 2.csv'
   num_private_person       : total=62  mean/row=0.09
   num_private_address      : total=136  mean/row=0.19
   num_private_email        : total=86  mean/row=0.12
   num_private_phone        : total=99  mean/row=0.14
   num_private_url          : total=77  mean/row=0.11
   num_private_date         : total=55  mean/row=0.08
   num_account_number       : total=123  mean/row=0.18
   num_secret               : total=121  mean/row=0.17


In [7]:
# CELL 5 — Load model  (pipeline imported HERE, not at top)
from transformers import pipeline
 
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
 
print(f"Loading: {MODEL_ID} ...")
try:
    pii_pipeline = pipeline(
        task="token-classification",
        model=MODEL_ID,
        trust_remote_code=True,
        aggregation_strategy=AGGREGATION,
        device=DEVICE,
        torch_dtype=TORCH_DTYPE,
    )
    print("✅ Model loaded successfully.")
except Exception as e:
    print(f"❌ GPU load failed: {e}")
    print("Retrying on CPU with float32 ...")
    DEVICE      = -1
    TORCH_DTYPE = torch.float32
    pii_pipeline = pipeline(
        task="token-classification",
        model=MODEL_ID,
        trust_remote_code=True,
        aggregation_strategy=AGGREGATION,
        device=DEVICE,
        torch_dtype=TORCH_DTYPE,
    )
    print("✅ Model loaded on CPU.")
 
# Smoke test
_test = pii_pipeline("My name is Alice Smith, email alice@example.com")
for s in _test:
    print(s['entity_group'], "|", s['word'], "|", f"{s['score']:.4f}")

Loading: openai/privacy-filter ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/140 [00:00<?, ?it/s]

✅ Model loaded successfully.
private_person |  Alice | 1.0000
private_person |  Smith | 1.0000
private_email |  alice@example | 1.0000
private_email | .com | 0.9906


In [8]:
# CELL 6 — Manual test (edit test_text freely)
# ════════════════════════════════════════════════════════════════
test_text = """My email id is abhijayduggal@gmail.com"""
 
def redact(text: str, spans: list, label_key: str = "entity_group") -> str:
    for span in sorted(spans, key=lambda s: s["start"], reverse=True):
        text = text[:span["start"]] + mask_token(span[label_key]) + text[span["end"]:]
    return text
 
def to_bioes(raw: str, spans: list, label_key: str = "entity_group") -> list:
    """Convert spans to BIOES token-level tags. Returns list of (token_str, tag)."""
    tokens = list(re.finditer(r"\S+", raw))
    labels = ["O"] * len(tokens)
    for span in sorted(spans, key=lambda s: s["start"]):
        in_span_idxs = [
            wi for wi, wm in enumerate(tokens)
            if wm.start() < span["end"] and wm.end() > span["start"]
        ]
        if not in_span_idxs:
            continue
        lbl = span[label_key]
        if len(in_span_idxs) == 1:
            labels[in_span_idxs[0]] = f"S-{lbl}"
        else:
            labels[in_span_idxs[0]]  = f"B-{lbl}"
            labels[in_span_idxs[-1]] = f"E-{lbl}"
            for idx in in_span_idxs[1:-1]:
                labels[idx] = f"I-{lbl}"
    return [(tokens[i].group(), labels[i]) for i in range(len(tokens))]
 
_raw_spans_manual = pii_pipeline(test_text)
_raw_spans_manual = [{**s, "score": float(s["score"])} for s in _raw_spans_manual]
 
# — normalised version
def _merge_spans_simple(spans, raw_text):
    if not spans: return []
    spans = sorted(spans, key=lambda s: s["start"])
    merged = [dict(spans[0])]
    for s in spans[1:]:
        prev = merged[-1]
        if s["entity_group"] == prev["entity_group"] and s["start"] <= prev["end"] + 1:
            prev["end"]  = max(prev["end"], s["end"])
            prev["word"] = raw_text[prev["start"]:prev["end"]]
        else:
            merged.append(dict(s))
    return merged
 
def _norm_ws(spans, raw_text):
    result = []
    for s in spans:
        st, en = s["start"], s["end"]
        while st < en and raw_text[st]   == " ": st += 1
        while en > st and raw_text[en-1] == " ": en -= 1
        result.append({**s, "start": st, "end": en, "word": raw_text[st:en]})
    return result
 
_norm_spans  = _norm_ws(_merge_spans_simple(_raw_spans_manual, test_text), test_text)
_masked      = redact(test_text, _raw_spans_manual)   # raw redacted
_norm_masked = redact(test_text, _norm_spans)
 
# ✅ FIX: use raw spans for raw BIOES, norm spans for norm BIOES — separately
_bioes_raw  = to_bioes(test_text, _raw_spans_manual)  # raw model output tags
_bioes_norm = to_bioes(test_text, _norm_spans)         # after merge+ws
 
W = 68
print("═" * W)
print(f"  INPUT  : {test_text}")
print("═" * W)
 
# Raw detected spans
if _raw_spans_manual:
    print(f"\n  RAW DETECTED SPANS  ({len(_raw_spans_manual)} found)")
    print(f"  {'#':<4} {'Label':<22} {'Text':<30} {'Score':>7}  {'Chars':>10}")
    print("  " + "─" * 62)
    for idx, s in enumerate(_raw_spans_manual, 1):
        char_range = f"[{s['start']}:{s['end']}]"
        print(f"  {idx:<4} {s['entity_group']:<22} {repr(s['word'])[:28]:<30} {s['score']:>7.4f}  {char_range:>10}")
else:
    print("  ⚠️  No PII detected.")
 
# Normalised spans (if different)
if len(_norm_spans) != len(_raw_spans_manual) or _norm_masked != _masked:
    print(f"\n  NORMALIZED SPANS  ({len(_norm_spans)} after merge+ws-transfer)")
    print(f"  {'#':<4} {'Label':<22} {'Text':<30} {'Chars':>10}")
    print("  " + "─" * 62)
    for idx, s in enumerate(_norm_spans, 1):
        char_range = f"[{s['start']}:{s['end']}]"
        print(f"  {idx:<4} {s['entity_group']:<22} {repr(s['word'])[:28]:<30} {char_range:>10}")
 
# BIOES token table — raw vs norm side by side
# ✅ FIX: _bioes_raw uses raw model spans, _bioes_norm uses normalized spans
print(f"\n  BIOES TOKEN TAGS")
print(f"  {'Token':<35} {'Raw Tag':<30} {'Norm Tag':<30}  Note")
print("  " + "─" * 100)
for (tok, raw_tag), (_, norm_tag) in zip(_bioes_raw, _bioes_norm):
    raw_disp  = raw_tag  if raw_tag  != "O" else "·"
    norm_disp = norm_tag if norm_tag != "O" else "·"
    note = "◄ merged/normalized" if raw_tag != norm_tag else ""
    print(f"  {tok:<35} {raw_disp:<30} {norm_disp:<30}  {note}")
 
print()
print(f"  RAW    MASKED : {_masked}")
print(f"  NORM   MASKED : {_norm_masked}")
print("═" * W)

════════════════════════════════════════════════════════════════════
  INPUT  : My email id is abhijayduggal@gmail.com
════════════════════════════════════════════════════════════════════

  RAW DETECTED SPANS  (2 found)
  #    Label                  Text                             Score       Chars
  ──────────────────────────────────────────────────────────────
  1    private_email          ' abhijayduggal@gmail'          1.0000     [14:34]
  2    private_email          '.com'                          0.9991     [34:38]

  NORMALIZED SPANS  (1 after merge+ws-transfer)
  #    Label                  Text                                Chars
  ──────────────────────────────────────────────────────────────
  1    private_email          'abhijayduggal@gmail.com'         [15:38]

  BIOES TOKEN TAGS
  Token                               Raw Tag                        Norm Tag                        Note
  ─────────────────────────────────────────────────────────────────────────────────────

In [9]:
# CELL 7 — Batch inference (BATCH_SIZE=1, safe for this model)
import transformers
transformers.logging.set_verbosity_error()
 
all_raw_spans = []   # raw model output, untouched
 
for text_i in tqdm(texts, desc="Inference"):
    spans = pii_pipeline(text_i)
    if spans and isinstance(spans[0], list):
        spans = spans[0]
    spans = [{**s, "score": float(s["score"])} for s in spans]
    all_raw_spans.append(spans)
 
# restore normal logging after inference
transformers.logging.set_verbosity_warning()
print(f"✅ Done — {len(all_raw_spans):,} texts processed.")

Inference:   0%|          | 0/700 [00:00<?, ?it/s]

✅ Done — 700 texts processed.


In [10]:
# CELL 8 — Core rule-based pipeline functions 
_MASK_RE = re.compile(r"\[([A-Z0-9_]+)\]")
 
# ── 8A: GT reconstruction ────────────────────────────────────
def reconstruct_gt_spans(raw: str, masked: str) -> list:
    """
    Rule-based alignment of maskedText against rawText.
    Recovers label, text, start, end for every placeholder.
    Handles multiple entities, repeated placeholders, adversarial spacing.
    """
    spans    = []
    raw_ptr  = 0
    mask_ptr = 0
 
    while mask_ptr < len(masked):
        m = _MASK_RE.match(masked, mask_ptr)
        if m:
            label    = m.group(1).lower()
            mask_ptr = m.end()
            # find literal text that follows this placeholder
            next_m        = _MASK_RE.search(masked, mask_ptr)
            literal_after = masked[mask_ptr : next_m.start()] if next_m else masked[mask_ptr:]
 
            if literal_after:
                idx      = raw.find(literal_after, raw_ptr)
                span_end = idx if idx != -1 else raw_ptr + 1
            else:
                span_end = len(raw)
 
            spans.append({
                "label" : label,
                "text"  : raw[raw_ptr:span_end],
                "start" : raw_ptr,
                "end"   : span_end,
            })
            raw_ptr = span_end
        else:
            raw_ptr  += 1
            mask_ptr += 1
 
    return spans

# ── 8B: Span merging (step 1 of normalization) ───────────────
def merge_spans(spans: list, raw_text: str, label_key: str = "entity_group") -> list:
    """
    Merge spans with identical labels that overlap, touch, or are
    separated by exactly 1 character. Merged text is ALWAYS reconstructed
    from rawText offsets — never by concatenating prediction strings.
    Tracks merge metadata for fragmentation analysis.
    """
    if not spans:
        return []
 
    spans  = sorted(spans, key=lambda s: s["start"])
    merged = [dict(spans[0])]
    merged[-1].setdefault("_merge_count", 1)
 
    for s in spans[1:]:
        prev      = merged[-1]
        same_lbl  = s[label_key] == prev[label_key]
        touches   = s["start"] <= prev["end"] + 1   # overlap OR adjacent OR gap=1
 
        if same_lbl and touches:
            prev["end"]          = max(prev["end"], s["end"])
            prev["text"]         = raw_text[prev["start"] : prev["end"]]
            prev["word"]         = prev["text"]
            prev["_merge_count"] = prev.get("_merge_count", 1) + 1
        else:
            ns = dict(s)
            ns.setdefault("_merge_count", 1)
            merged.append(ns)
 
    return merged

# ── 8C: Whitespace transfer (step 2 of normalization) ────────
def normalize_whitespace(spans: list, raw_text: str) -> list:
    """
    Transfer leading/trailing whitespace OUTSIDE the span boundary.
    Never removes whitespace — moves it so masked output stays readable.
    Applied ONLY AFTER merging.
    """
    result = []
    for s in spans:
        start, end = s["start"], s["end"]
        had_ws     = False
 
        while start < end and raw_text[start] == " ":
            start += 1
            had_ws = True
        while end > start and raw_text[end - 1] == " ":
            end   -= 1
            had_ws = True
 
        ns         = dict(s)
        ns["start"]             = start
        ns["end"]               = end
        ns["text"]              = raw_text[start:end]
        ns["word"]              = raw_text[start:end]
        ns["_had_ws_transfer"]  = had_ws
        result.append(ns)
    return result

# ── 8D: Span-level evaluation (strict exact match) ───────────
def span_eval(gt_spans: list, pred_spans: list, pred_label_key: str = "entity_group"):
    """
    A prediction is correct ONLY IF label + start + end all match exactly.
    """
    gt_set   = {(s["label"],        s["start"], s["end"]) for s in gt_spans}
    pred_set = {(s[pred_label_key], s["start"], s["end"]) for s in pred_spans}
    tp = len(gt_set & pred_set)
    fp = len(pred_set - gt_set)
    fn = len(gt_set  - pred_set)
    p  = tp / (tp + fp) if (tp + fp) else 0.0
    r  = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return tp, fp, fn, p, r, f1

# ── 8E: Token-level evaluation ───────────────────────────────
def token_eval(raw_text: str, gt_spans: list, pred_spans: list,
               pred_label_key: str = "entity_group"):
    """
    Tokenise by whitespace. Label each token with its entity if it
    falls inside a span. Compute TP/FP/FN at token level.
    """
    tokens = list(re.finditer(r"\S+", raw_text))
 
    def label_tokens(spans, key):
        labels  = ["O"] * len(tokens)
        in_span = False
        for span in sorted(spans, key=lambda s: s["start"]):
            in_span = False
            for wi, wm in enumerate(tokens):
                if wm.start() < span["end"] and wm.end() > span["start"]:
                    labels[wi] = span[key]
                    in_span    = True
                elif in_span:
                    break
        return labels
 
    gt_lbl   = label_tokens(gt_spans,   "label")
    pred_lbl = label_tokens(pred_spans, pred_label_key)
 
    tp = sum(1 for g, p in zip(gt_lbl, pred_lbl) if g != "O" and g == p)
    fp = sum(1 for g, p in zip(gt_lbl, pred_lbl) if p != "O" and g != p)
    fn = sum(1 for g, p in zip(gt_lbl, pred_lbl) if g != "O" and p == "O")
    pr = tp / (tp + fp) if (tp + fp) else 0.0
    rc = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * pr * rc / (pr + rc) if (pr + rc) else 0.0
    return tp, fp, fn, pr, rc, f1

# ── 8F: Category helpers ─────────────────────────────────────
def get_label_set_from_masked(text: str) -> set:
    return set(_MASK_RE.findall(text))
 
print("✅ All pipeline functions defined.")

✅ All pipeline functions defined.


In [11]:
print(df.columns.tolist())
print(repr(df.columns[0]))
print(COL_SNO)
print(repr(COL_SNO))

['s_no', 'rawText', 'maskedText', 'numberOfMasks', 'num_private_person', 'num_private_address', 'num_private_email', 'num_private_phone', 'num_private_url', 'num_private_date', 'num_account_number', 'num_secret', 'category', 'difficulty', 'adversarial_type', 'trick_reason']
's_no'
s_no
's_no'


In [12]:
# CELL 9 — Per-row evaluation + build output dataframe
# ════════════════════════════════════════════════════════════════
 
# GT reconstruction
gt_spans_all = [reconstruct_gt_spans(r, m) for r, m in zip(texts, ground_truths)]
print(f"GT spans total: {sum(len(s) for s in gt_spans_all):,}")
 
# Accumulators for dataset-level micro-avg
acc = {k: {"raw": [0,0,0], "norm": [0,0,0], "tok": [0,0,0]} for k in ["tp","fp","fn"]}
# Per-label accumulators
per_label_raw  = {lbl: [0,0,0] for lbl in ALL_LABELS}  # tp,fp,fn
per_label_norm = {lbl: [0,0,0] for lbl in ALL_LABELS}
 
# Global counters
total_fragments = 0
total_merges    = 0
total_ws        = 0
missed_counts        = Counter()
hallucinated_counts  = Counter()
 
rows_out = []
 
for i, (raw, gt_masked, raw_spans, gt_spans) in enumerate(
        zip(texts, ground_truths, all_raw_spans, gt_spans_all)):
 
    # ── Normalize predictions ──────────────────────────────
    norm_spans_merged = merge_spans(raw_spans, raw)
    norm_spans        = normalize_whitespace(norm_spans_merged, raw)
 
    # ── Fragmentation metadata ─────────────────────────────
    spans_merged_count  = sum(s.get("_merge_count", 1) - 1 for s in norm_spans)
    had_fragmentation   = spans_merged_count > 0
    had_ws_transfer     = any(s.get("_had_ws_transfer", False) for s in norm_spans)
 
    # check overlap vs adjacency in raw (before merge)
    had_overlap  = False
    had_adjacent = False
    raw_sorted   = sorted(raw_spans, key=lambda s: s["start"])
    for j in range(1, len(raw_sorted)):
        prev_s = raw_sorted[j-1]
        cur_s  = raw_sorted[j]
        if prev_s["entity_group"] == cur_s["entity_group"]:
            gap = cur_s["start"] - prev_s["end"]
            if gap < 0:
                had_overlap  = True
            elif gap <= 1:
                had_adjacent = True
 
    if had_fragmentation: total_fragments += 1
    if spans_merged_count > 0: total_merges += spans_merged_count
    if had_ws_transfer: total_ws += 1
 
    # ── Masked text outputs ────────────────────────────────
    raw_masked_text  = redact(raw, raw_spans)
    norm_masked_text = redact(raw, norm_spans)
 
    # ── Span-level eval ────────────────────────────────────
    r_tp, r_fp, r_fn, r_p, r_r, r_f1 = span_eval(gt_spans, raw_spans)
    n_tp, n_fp, n_fn, n_p, n_r, n_f1 = span_eval(gt_spans, norm_spans)
 
    # ── Token-level eval ───────────────────────────────────
    t_tp, t_fp, t_fn, t_p, t_r, t_f1 = token_eval(raw, gt_spans, norm_spans)
 
    # ── Per-label span eval (normalized) ──────────────────
    for lbl in ALL_LABELS:
        gt_lbl   = [s for s in gt_spans   if s["label"]        == lbl]
        pred_lbl = [s for s in norm_spans  if s["entity_group"] == lbl]
        ltp, lfp, lfn, _, _, _ = span_eval(gt_lbl, pred_lbl)
        per_label_norm[lbl][0] += ltp
        per_label_norm[lbl][1] += lfp
        per_label_norm[lbl][2] += lfn
 
        gt_lbl_r   = [s for s in gt_spans  if s["label"]        == lbl]
        pred_lbl_r = [s for s in raw_spans if s["entity_group"] == lbl]
        ltp_r, lfp_r, lfn_r, _, _, _ = span_eval(gt_lbl_r, pred_lbl_r)
        per_label_raw[lbl][0] += ltp_r
        per_label_raw[lbl][1] += lfp_r
        per_label_raw[lbl][2] += lfn_r
 
    # ── Accumulate dataset-level totals ───────────────────
    for vals, key in [([r_tp,r_fp,r_fn],"raw"), ([n_tp,n_fp,n_fn],"norm"), ([t_tp,t_fp,t_fn],"tok")]:
        acc["tp"][key][0] += vals[0]
        acc["fp"][key][0] += vals[1]
        acc["fn"][key][0] += vals[2]
 
    # ── Category analysis ──────────────────────────────────
    gt_cats   = get_label_set_from_masked(gt_masked)
    pred_cats = get_label_set_from_masked(norm_masked_text)
    correct_cats      = gt_cats & pred_cats
    missed_cats       = gt_cats - pred_cats
    hallucinated_cats = pred_cats - gt_cats
 
    for lbl in missed_cats:       missed_counts[lbl]       += 1
    for lbl in hallucinated_cats: hallucinated_counts[lbl] += 1
 
    # ── Label-agnostic coverage ────────────────────────────────
    # Was each GT span covered by ANY prediction, regardless of label?
    # A prediction "covers" a GT span if it overlaps by >50% of the GT span length.
    def agnostic_coverage(gt_spans, pred_spans):
        """
        For each GT span: check if ANY pred span overlaps it by at least 50%,
        ignoring labels entirely. Measures pure PII detection regardless of category.
        Returns tp, fp, fn and precision/recall/f1.
        """
        gt_covered  = []
        pred_matched = [False] * len(pred_spans)
        for g in gt_spans:
            g_len   = max(g["end"] - g["start"], 1)
            covered = False
            for pi, p in enumerate(pred_spans):
                overlap = max(0, min(g["end"], p["end"]) - max(g["start"], p["start"]))
                if overlap / g_len >= 0.5:
                    covered = True
                    pred_matched[pi] = True
            gt_covered.append(covered)
        tp = sum(gt_covered)
        fn = sum(1 for c in gt_covered if not c)
        fp = sum(1 for m in pred_matched if not m)
        p  = tp / (tp + fp) if (tp + fp) else 0.0
        r  = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2*p*r/(p+r)   if (p + r)   else 0.0
        return tp, fp, fn, p, r, f1
 
    ac_tp, ac_fp, ac_fn, ac_p, ac_r, ac_f1 = agnostic_coverage(gt_spans, norm_spans)
    scores = [s["score"] for s in raw_spans if "score" in s]
    avg_conf = float(np.mean(scores))  if scores else 0.0
    max_conf = float(np.max(scores))   if scores else 0.0
 
    # ── Entity analysis strings ───────────────────────────
    gt_set_str   = {(s["label"],        s["start"], s["end"]) for s in gt_spans}
    norm_set_str = {(s["entity_group"], s["start"], s["end"]) for s in norm_spans}
 
    missed_ents       = [s for s in gt_spans  if (s["label"],        s["start"],s["end"]) not in norm_set_str]
    fp_ents           = [s for s in norm_spans if (s["entity_group"],s["start"],s["end"]) not in gt_set_str]
    fragmented_ents   = [s for s in norm_spans if s.get("_merge_count",1) > 1]
    merged_ent_labels = [s["entity_group"] for s in norm_spans if s.get("_merge_count",1) > 1]
 
    rows_out.append({
        # original input
        "s_no"             : df.at[i, COL_SNO],
        "rawText"          : raw,
        "maskedText"       : gt_masked,
        "numberOfMasks"    : df.at[i, "numberOfMasks"],
        "category"         : df.at[i, COL_CATEGORY],
        "difficulty"       : df.at[i, COL_DIFFICULTY],
        "adversarial_type" : df.at[i, COL_ADVERSARIAL],
        "trick_reason"     : df.at[i, "trick_reason"],
        # GT structure
        "gt_spans"         : json.dumps([{"label":s["label"],"text":s["text"],"start":s["start"],"end":s["end"]} for s in gt_spans]),
        "gt_span_count"    : len(gt_spans),
        # raw predictions
        "raw_predicted_spans"  : json.dumps([{**s} for s in raw_spans]),
        "raw_predicted_count"  : len(raw_spans),
        # normalized predictions
        "normalized_predicted_spans" : json.dumps([{"entity_group":s["entity_group"],"text":s["text"],"start":s["start"],"end":s["end"]} for s in norm_spans]),
        "normalized_predicted_count" : len(norm_spans),
        # normalization insights
        "spans_merged_count"    : spans_merged_count,
        "had_fragmentation"     : had_fragmentation,
        "had_overlap"           : had_overlap,
        "had_adjacent_merge"    : had_adjacent,
        "had_whitespace_transfer": had_ws_transfer,
        # raw span metrics
        "raw_span_tp" : r_tp, "raw_span_fp": r_fp, "raw_span_fn": r_fn,
        "raw_span_precision": round(r_p, 4),
        "raw_span_recall"   : round(r_r, 4),
        "raw_span_f1"       : round(r_f1, 4),
        # normalized span metrics
        "norm_span_tp": n_tp, "norm_span_fp": n_fp, "norm_span_fn": n_fn,
        "norm_span_precision": round(n_p, 4),
        "norm_span_recall"   : round(n_r, 4),
        "norm_span_f1"       : round(n_f1, 4),
        # token metrics
        "token_tp": t_tp, "token_fp": t_fp, "token_fn": t_fn,
        "token_precision": round(t_p, 4),
        "token_recall"   : round(t_r, 4),
        "token_f1"       : round(t_f1, 4),
        # entity analysis
        "missed_entities"      : ", ".join(f"{s['label']}({s['start']},{s['end']})" for s in missed_ents),
        "false_positive_entities": ", ".join(f"{s['entity_group']}({s['start']},{s['end']})" for s in fp_ents),
        "fragmented_entities"  : ", ".join(merged_ent_labels),
        "merged_entities"      : spans_merged_count,
        # category analysis
        "gt_mask_categories"          : ", ".join(sorted(gt_cats)),
        "predicted_mask_categories"   : ", ".join(sorted(pred_cats)),
        "correct_categories"          : ", ".join(sorted(correct_cats)),
        "missed_categories"           : ", ".join(sorted(missed_cats)),
        "hallucinated_categories"     : ", ".join(sorted(hallucinated_cats)),
        "all_correct"                 : missed_cats == set() and hallucinated_cats == set(),
        "any_missed"                  : len(missed_cats) > 0,
        "any_hallucinated"            : len(hallucinated_cats) > 0,
        # agnostic coverage (PII masked regardless of label)
        "agnostic_tp"        : ac_tp,
        "agnostic_fp"        : ac_fp,
        "agnostic_fn"        : ac_fn,
        "agnostic_precision" : round(ac_p,  4),
        "agnostic_recall"    : round(ac_r,  4),
        "agnostic_f1"        : round(ac_f1, 4),
        # confidence
        "average_prediction_confidence": round(avg_conf, 4),
        "max_prediction_confidence"    : round(max_conf, 4),
        # text outputs
        "raw_masked_text"        : raw_masked_text,
        "normalized_masked_text" : norm_masked_text,
    })
 
df_out = pd.DataFrame(rows_out)
 
# save (retry with timestamp if file is open in Excel)
try:
    df_out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"✅ Saved → {OUTPUT_CSV}")
except PermissionError:
    import time
    alt = OUTPUT_CSV.replace(".csv", f"_{int(time.time())}.csv")
    df_out.to_csv(alt, index=False, encoding="utf-8-sig")
    print(f"⚠️  File locked — saved to → {alt}")
 
print(f"   Rows: {len(df_out):,}  |  Columns: {len(df_out.columns)}")

GT spans total: 759
✅ Saved → predictions/predictions_FP_hin.csv
   Rows: 700  |  Columns: 59


In [13]:
# CELL 10 — Dataset-level micro-averaged metrics
# ════════════════════════════════════════════════════════════════
def micro_prf(tp_list, fp_list, fn_list):
    tp = sum(x[0] for x in tp_list)
    fp = sum(x[0] for x in fp_list)
    fn = sum(x[0] for x in fn_list)
    p  = tp / (tp + fp) if (tp + fp) else 0.0
    r  = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2*p*r/(p+r)    if (p  + r)  else 0.0
    return tp, fp, fn, p, r, f1
 
# unpack accumulators
def unpack(key):
    return acc["tp"][key], acc["fp"][key], acc["fn"][key]
 
raw_tp, raw_fp, raw_fn   = df_out["raw_span_tp"].sum(), df_out["raw_span_fp"].sum(), df_out["raw_span_fn"].sum()
norm_tp, norm_fp, norm_fn = df_out["norm_span_tp"].sum(), df_out["norm_span_fp"].sum(), df_out["norm_span_fn"].sum()
tok_tp, tok_fp, tok_fn   = df_out["token_tp"].sum(), df_out["token_fp"].sum(), df_out["token_fn"].sum()
 
def prf(tp, fp, fn):
    p  = tp/(tp+fp) if (tp+fp) else 0.0
    r  = tp/(tp+fn) if (tp+fn) else 0.0
    f1 = 2*p*r/(p+r) if (p+r) else 0.0
    return p, r, f1
 
raw_p,  raw_r,  raw_f1  = prf(raw_tp,  raw_fp,  raw_fn)
norm_p, norm_r, norm_f1 = prf(norm_tp, norm_fp, norm_fn)
tok_p,  tok_r,  tok_f1  = prf(tok_tp,  tok_fp,  tok_fn)
 
# Label-agnostic coverage — dataset level
ac_tp_total = df_out["agnostic_tp"].sum()
ac_fp_total = df_out["agnostic_fp"].sum()
ac_fn_total = df_out["agnostic_fn"].sum()
ac_p_total, ac_r_total, ac_f1_total = prf(ac_tp_total, ac_fp_total, ac_fn_total)
 
total          = len(df_out)
total_gt       = df_out["gt_span_count"].sum()
total_raw_pred = df_out["raw_predicted_count"].sum()
total_norm_pred= df_out["normalized_predicted_count"].sum()
frag_cases     = df_out["had_fragmentation"].sum()
ws_cases       = df_out["had_whitespace_transfer"].sum()
total_merged   = df_out["merged_entities"].sum()
total_fp_rows  = df_out["any_hallucinated"].sum()
total_fn_rows  = df_out["any_missed"].sum()
cat_all_correct= df_out["all_correct"].sum()
 
# Per-label metrics
per_label_results = {}
for lbl in ALL_LABELS:
    tp_n, fp_n, fn_n = per_label_norm[lbl]
    tp_r, fp_r, fn_r = per_label_raw[lbl]
    p_n, r_n, f1_n   = prf(tp_n, fp_n, fn_n)
    p_r, r_r, f1_r   = prf(tp_r, fp_r, fn_r)
    per_label_results[lbl] = {
        "norm_tp":tp_n,"norm_fp":fp_n,"norm_fn":fn_n,
        "norm_precision":round(p_n,4),"norm_recall":round(r_n,4),"norm_f1":round(f1_n,4),
        "raw_tp":tp_r,"raw_fp":fp_r,"raw_fn":fn_r,
        "raw_precision":round(p_r,4),"raw_recall":round(r_r,4),"raw_f1":round(f1_r,4),
        "missed_rows":missed_counts.get(lbl.upper(),0),
        "hallucinated_rows":hallucinated_counts.get(lbl.upper(),0),
        "gt_rows":int((df[LABEL_TO_COL[lbl]] > 0).sum()) if LABEL_TO_COL[lbl] in df.columns else 0,
    }
 
# Fragmentation analysis
frag_rate      = frag_cases / total if total else 0.0
avg_frags      = total_merged / frag_cases if frag_cases else 0.0
merge_success  = norm_tp / raw_tp if raw_tp else 0.0   # normalized TP / raw TP
 
print("✅ Dataset-level metrics computed.")

✅ Dataset-level metrics computed.


In [14]:
# CELL 11 — Slice report (by category / difficulty / adversarial)
def slice_report(col):
    if col not in df_out.columns:
        print(f"  ('{col}' not found)"); return
    print(f"\n── Slice by '{col}' ───────────────────────────────────────")
    grp = df_out.groupby(col).agg(
        rows             = ("s_no",               "count"),
        norm_f1          = ("norm_span_f1",        "mean"),
        token_f1         = ("token_f1",            "mean"),
        any_missed_pct   = ("any_missed",          lambda x: x.mean()*100),
        any_halluc_pct   = ("any_hallucinated",    lambda x: x.mean()*100),
        all_correct_pct  = ("all_correct",         lambda x: x.mean()*100),
        avg_gt_spans     = ("gt_span_count",       "mean"),
        avg_pred_spans   = ("normalized_predicted_count", "mean"),
        frag_pct         = ("had_fragmentation",   lambda x: x.mean()*100),
    ).round(3)
    print(grp.to_string())
 
for col in [COL_CATEGORY, COL_DIFFICULTY, COL_ADVERSARIAL]:
    slice_report(col)


── Slice by 'category' ───────────────────────────────────────
                 rows  norm_f1  token_f1  any_missed_pct  any_halluc_pct  all_correct_pct  avg_gt_spans  avg_pred_spans  frag_pct
category                                                                                                                         
account_number    117    0.325     0.375          58.120          37.607           38.462         1.000           1.034    56.410
mixed              57    0.223     0.266          87.719          15.789           10.526         2.000           0.912    49.123
private_address   132    0.125     0.583          25.758          12.121           64.394         1.015           1.250    62.879
private_date       52    0.798     0.862           3.846           5.769           90.385         1.000           1.288    88.462
private_email      80    0.910     0.920           0.000           5.000           95.000         1.000           1.200    93.750
private_person      5    0

In [15]:
# CELL 12 — FINAL SCORECARD
# ════════════════════════════════════════════════════════════════
W = 66
 
def row(left, right="", fill="─"):
    inner = f"  {left:<45} {right:>14}  "
    return f"║{inner}║"
 
print("╔" + "═"*W + "╗")
print(f"║{'  FINAL SCORECARD — openai/privacy-filter':^{W}}║")
print("╠" + "═"*W + "╣")
print(f"║{'  Dataset':^{W}}║")
print("╠" + "─"*W + "╣")
print(row(f"Total rows", str(total)))
print(row(f"GT spans",   str(int(total_gt))))
print(row(f"Raw predicted spans",        str(int(total_raw_pred))))
print(row(f"Normalized predicted spans", str(int(total_norm_pred))))
print("╠" + "═"*W + "╣")
print(f"║{'  Span-level Metrics':^{W}}║")
print("╠" + "─"*W + "╣")
print(row("RAW   Precision / Recall / F1",  f"{raw_p:.4f} / {raw_r:.4f} / {raw_f1:.4f}"))
print(row("NORM  Precision / Recall / F1",  f"{norm_p:.4f} / {norm_r:.4f} / {norm_f1:.4f}"))
print("╠" + "═"*W + "╣")
print(f"║{'  Token-level Metrics':^{W}}║")
print("╠" + "─"*W + "╣")
print(row("Token Precision / Recall / F1",  f"{tok_p:.4f} / {tok_r:.4f} / {tok_f1:.4f}"))
print("╠" + "═"*W + "╣")
print(f"║{'  Label-Agnostic Coverage  (PII masked regardless of label)':^{W}}║")
print("╠" + "─"*W + "╣")
print(row("Agnostic Precision / Recall / F1", f"{ac_p_total:.4f} / {ac_r_total:.4f} / {ac_f1_total:.4f}"))
print(row("  TP (GT spans covered by any pred)",  str(int(ac_tp_total))))
print(row("  FP (preds covering no GT span)",     str(int(ac_fp_total))))
print(row("  FN (GT spans not covered at all)",   str(int(ac_fn_total))))
print("╠" + "═"*W + "╣")
print(f"║{'  Category-level':^{W}}║")
print("╠" + "─"*W + "╣")
print(row("All categories correct",  f"{cat_all_correct}/{total}  ({cat_all_correct/total*100:.1f}%)"))
print(row("Any missed",              f"{total_fn_rows}/{total}  ({total_fn_rows/total*100:.1f}%)"))
print(row("Any hallucinated",        f"{total_fp_rows}/{total}  ({total_fp_rows/total*100:.1f}%)"))
print("╠" + "═"*W + "╣")
print(f"║{'  Fragmentation Analysis':^{W}}║")
print("╠" + "─"*W + "╣")
print(row("Rows with fragmentation",      f"{frag_cases}/{total}  ({frag_rate*100:.1f}%)"))
print(row("Total spans merged",           str(int(total_merged))))
print(row("Avg fragments per merge-case", f"{avg_frags:.2f}"))
print(row("Rows with whitespace transfer",str(int(ws_cases))))
print(row("Merge success rate (TP gain)", f"{merge_success:.4f}"))
print("╠" + "═"*W + "╣")
print(f"║{'  Per-Label (Normalized Span)':^{W}}║")
print("╠" + "─"*W + "╣")
# header
hdr = f"  {'Label':<20} {'P':>6} {'R':>6} {'F1':>6}  {'Missed':>12}  {'Halluc':>12}"
print(f"║{hdr:<{W}}║")
print("║" + "─"*W + "║")
for lbl in ALL_LABELS:
    r_      = per_label_results[lbl]
    gt_rows = r_["gt_rows"]
    m       = r_["missed_rows"]
    h       = r_["hallucinated_rows"]
    # pred rows for this label
    pred_rows = sum(1 for _, row_ in df_out.iterrows()
                    if lbl.upper() in row_["predicted_mask_categories"])
    m_str   = f"{m}/{gt_rows}"
    h_str   = f"{h}/{pred_rows}"
    line    = f"  {lbl:<20} {r_['norm_precision']:>6.4f} {r_['norm_recall']:>6.4f} {r_['norm_f1']:>6.4f}  {m_str:>12}  {h_str:>12}"
    print(f"║{line:<{W}}║")
print("╚" + "═"*W + "╝")
print(f"\n✅ Done. Output saved to: {OUTPUT_CSV}")
 
# ── Save summary JSON ──────────────────────────────────────────
summary = {
    "dataset": {
        "total_rows": int(total),
        "total_gt_entities": int(total_gt),
        "total_predicted_raw": int(total_raw_pred),
        "total_predicted_normalized": int(total_norm_pred),
    },
    "raw_span":  {"precision":round(raw_p,4),"recall":round(raw_r,4),"f1":round(raw_f1,4),
                  "tp":int(raw_tp),"fp":int(raw_fp),"fn":int(raw_fn)},
    "norm_span": {"precision":round(norm_p,4),"recall":round(norm_r,4),"f1":round(norm_f1,4),
                  "tp":int(norm_tp),"fp":int(norm_fp),"fn":int(norm_fn)},
    "token":     {"precision":round(tok_p,4),"recall":round(tok_r,4),"f1":round(tok_f1,4),
                  "tp":int(tok_tp),"fp":int(tok_fp),"fn":int(tok_fn)},
    "agnostic_coverage": {"precision":round(ac_p_total,4),"recall":round(ac_r_total,4),
                           "f1":round(ac_f1_total,4),"tp":int(ac_tp_total),
                           "fp":int(ac_fp_total),"fn":int(ac_fn_total)},
    "fragmentation": {
        "fragmentation_rate": round(frag_rate,4),
        "avg_fragments_per_merge_case": round(avg_frags,4),
        "merge_success_rate": round(merge_success,4),
        "total_fragmented_cases": int(frag_cases),
        "total_merged_spans": int(total_merged),
        "total_whitespace_transfer_cases": int(ws_cases),
    },
    "category_level": {
        "all_correct": int(cat_all_correct),
        "any_missed": int(total_fn_rows),
        "any_hallucinated": int(total_fp_rows),
    },
    "per_label": per_label_results,
}
 
json_path = OUTPUT_CSV.replace(".csv", "_summary.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(f"✅ Summary JSON saved → {json_path}")

╔══════════════════════════════════════════════════════════════════╗
║              FINAL SCORECARD — openai/privacy-filter             ║
╠══════════════════════════════════════════════════════════════════╣
║                              Dataset                             ║
╠──────────────────────────────────────────────────────────────────╣
║  Total rows                                               700  ║
║  GT spans                                                 759  ║
║  Raw predicted spans                                     1025  ║
║  Normalized predicted spans                               593  ║
╠══════════════════════════════════════════════════════════════════╣
║                         Span-level Metrics                       ║
╠──────────────────────────────────────────────────────────────────╣
║  RAW   Precision / Recall / F1                 0.0088 / 0.0119 / 0.0101  ║
║  NORM  Precision / Recall / F1                 0.4132 / 0.3228 / 0.3624  ║
╠═════════════════════════

In [16]:
# CELL 13 — INSIGHT REPORT
# Deep-dive analytics beyond raw scores
# ════════════════════════════════════════════════════════════════
W2 = 72
 
def sec(title):
    print("╠" + "═"*W2 + "╣")
    print(f"║{'  ' + title:{W2}}║")
    print("╠" + "─"*W2 + "╣")
 
def irow(left, right=""):
    line = f"  {left:<48} {right:>18}"
    print(f"║{line:<{W2}}║")
 
print("╔" + "═"*W2 + "╗")
print(f"║{'  INSIGHT REPORT — openai/privacy-filter':^{W2}}║")
 
# ── 1. Confidence calibration ─────────────────────────────────
sec("1. Confidence Calibration")
irow("Metric", "Value")
print("║" + "─"*W2 + "║")
 
# split rows into high-conf (>=0.9) and low-conf (<0.9)
hi_mask  = df_out["average_prediction_confidence"] >= 0.9
lo_mask  = df_out["average_prediction_confidence"] <  0.9
hi_f1    = df_out.loc[hi_mask,  "norm_span_f1"].mean() if hi_mask.any()  else 0.0
lo_f1    = df_out.loc[lo_mask,  "norm_span_f1"].mean() if lo_mask.any()  else 0.0
hi_rows  = hi_mask.sum()
lo_rows  = lo_mask.sum()
irow(f"High-conf rows (avg_conf ≥ 0.9)  →  norm_span_f1", f"{hi_rows} rows / F1={hi_f1:.4f}")
irow(f"Low-conf  rows (avg_conf < 0.9)  →  norm_span_f1", f"{lo_rows} rows / F1={lo_f1:.4f}")
 
# confidence of correct vs wrong predictions at span level
tp_confs, fp_confs = [], []
for i, row_ in df_out.iterrows():
    gt_s    = json.loads(row_["gt_spans"])
    pred_s  = json.loads(row_["normalized_predicted_spans"])
    raw_s   = json.loads(row_["raw_predicted_spans"])
    gt_set_ = {(s["label"], s["start"], s["end"]) for s in gt_s}
    for ps in raw_s:
        key = (ps["entity_group"], ps["start"], ps["end"])
        if key in gt_set_:
            tp_confs.append(ps["score"])
        else:
            fp_confs.append(ps["score"])
avg_tp_conf = float(np.mean(tp_confs)) if tp_confs else 0.0
avg_fp_conf = float(np.mean(fp_confs)) if fp_confs else 0.0
irow("Avg confidence of TRUE POSITIVE predictions",  f"{avg_tp_conf:.4f}")
irow("Avg confidence of FALSE POSITIVE predictions", f"{avg_fp_conf:.4f}")
irow("Confidence gap (TP - FP)",                     f"{avg_tp_conf - avg_fp_conf:+.4f}")
 
# ── 2. Boundary error analysis ────────────────────────────────
sec("2. Boundary Error Analysis  (right label, wrong boundary)")
irow("Label", "Overlap-but-wrong  |  Avg start_err  |  Avg end_err")
print("║" + "─"*W2 + "║")
 
boundary_stats = {lbl: {"count":0,"start_errs":[],"end_errs":[]} for lbl in ALL_LABELS}
for i, row_ in df_out.iterrows():
    gt_s   = json.loads(row_["gt_spans"])
    norm_s = json.loads(row_["normalized_predicted_spans"])
    gt_set_ = {(s["label"], s["start"], s["end"]) for s in gt_s}
    for g in gt_s:
        for p in norm_s:
            if (p["entity_group"] == g["label"]
                    and not (p["start"] == g["start"] and p["end"] == g["end"])
                    and p["start"] < g["end"] and p["end"] > g["start"]):
                stats = boundary_stats[g["label"]]
                stats["count"]      += 1
                stats["start_errs"].append(p["start"] - g["start"])
                stats["end_errs"].append(p["end"]     - g["end"])
 
for lbl in ALL_LABELS:
    st = boundary_stats[lbl]
    if st["count"] == 0:
        irow(f"  {lbl}", "none")
    else:
        avg_se = np.mean(st["start_errs"])
        avg_ee = np.mean(st["end_errs"])
        irow(f"  {lbl}", f"{st['count']:>6} cases  | {avg_se:+.1f}  | {avg_ee:+.1f}")
 
# ── 3. Fragmentation by label ─────────────────────────────────
sec("3. Fragmentation by Label  (how often each label gets split)")
irow("Label", "Frag cases  |  Avg fragments  |  of GT rows")
print("║" + "─"*W2 + "║")
 
frag_by_label = {lbl: {"cases":0, "total_frags":0} for lbl in ALL_LABELS}
for i, row_ in df_out.iterrows():
    norm_s = json.loads(row_["normalized_predicted_spans"])
    for s in norm_s:
        lbl = s.get("entity_group","")
        if lbl in frag_by_label and s.get("_merge_count",1) > 1:
            frag_by_label[lbl]["cases"]       += 1
            frag_by_label[lbl]["total_frags"] += s.get("_merge_count",1)
 
for lbl in ALL_LABELS:
    fb  = frag_by_label[lbl]
    gt_ = per_label_results[lbl]["gt_rows"]
    if fb["cases"] == 0:
        irow(f"  {lbl}", "no fragmentation")
    else:
        avg_f = fb["total_frags"] / fb["cases"]
        pct   = fb["cases"] / gt_ * 100 if gt_ else 0
        irow(f"  {lbl}", f"{fb['cases']:>4} cases  |  avg {avg_f:.1f} frags  |  {pct:.1f}%")
 
# ── 4. Trick reason analysis ──────────────────────────────────
sec("4. Trick Reason Analysis  (which tricks fool the model most)")
irow("Trick Reason", "Rows  |  norm_f1  |  missed%  |  halluc%")
print("║" + "─"*W2 + "║")
 
if "trick_reason" in df_out.columns:
    trick_grp = df_out.groupby("trick_reason").agg(
        rows         = ("s_no",             "count"),
        norm_f1      = ("norm_span_f1",      "mean"),
        missed_pct   = ("any_missed",        lambda x: x.mean()*100),
        halluc_pct   = ("any_hallucinated",  lambda x: x.mean()*100),
    ).round(3).sort_values("norm_f1")
    for trick, trow in trick_grp.iterrows():
        summary_str = f"{int(trow['rows']):>3} rows | f1={trow['norm_f1']:.3f} | m={trow['missed_pct']:.0f}% | h={trow['halluc_pct']:.0f}%"
        irow(f"  {str(trick)[:40]}", summary_str)
else:
    irow("  (trick_reason column not found)", "")
 
# ── 5. Adversarial type deep-dive ─────────────────────────────
sec("5. Adversarial Type Deep-Dive")
irow("Adversarial Type", "Rows  |  norm_f1  |  token_f1  |  frag%")
print("║" + "─"*W2 + "║")
 
adv_grp = df_out.groupby(COL_ADVERSARIAL).agg(
    rows      = ("s_no",               "count"),
    norm_f1   = ("norm_span_f1",       "mean"),
    tok_f1    = ("token_f1",           "mean"),
    frag_pct  = ("had_fragmentation",  lambda x: x.mean()*100),
).round(3).sort_values("norm_f1")
for adv_type, arow in adv_grp.iterrows():
    summary_str = f"{int(arow['rows']):>3} | f1={arow['norm_f1']:.3f} | tok={arow['tok_f1']:.3f} | frag={arow['frag_pct']:.0f}%"
    irow(f"  {str(adv_type)[:38]}", summary_str)
 
# ── 6. Label co-occurrence errors ─────────────────────────────
sec("6. Label Confusion  (most common hallucinated label per GT label)")
irow("GT expected", "Most hallucinated instead / alongside")
print("║" + "─"*W2 + "║")
 
confusion = {lbl: Counter() for lbl in ALL_LABELS}
for i, row_ in df_out.iterrows():
    gt_cats_  = set(row_["gt_mask_categories"].split(", ")) if row_["gt_mask_categories"] else set()
    hall_cats = set(row_["hallucinated_categories"].split(", ")) if row_["hallucinated_categories"] else set()
    for g in gt_cats_:
        lbl_lower = g.lower()
        if lbl_lower in confusion:
            for h in hall_cats:
                confusion[lbl_lower][h] += 1
 
for lbl in ALL_LABELS:
    most_common = confusion[lbl].most_common(2)
    if most_common:
        val = "  ".join(f"{h}({c})" for h,c in most_common)
        irow(f"  {lbl}", val)
    else:
        irow(f"  {lbl}", "—")
 
print("╚" + "═"*W2 + "╝")
print("✅ Insight report complete.")

╔════════════════════════════════════════════════════════════════════════╗
║                  INSIGHT REPORT — openai/privacy-filter                ║
╠════════════════════════════════════════════════════════════════════════╣
║  1. Confidence Calibration                                             ║
╠────────────────────────────────────────────────────────────────────────╣
║  Metric                                                        Value   ║
║────────────────────────────────────────────────────────────────────────║
║  High-conf rows (avg_conf ≥ 0.9)  →  norm_span_f1 248 rows / F1=0.7062 ║
║  Low-conf  rows (avg_conf < 0.9)  →  norm_span_f1 452 rows / F1=0.1027 ║
║  Avg confidence of TRUE POSITIVE predictions                  0.9635   ║
║  Avg confidence of FALSE POSITIVE predictions                 0.8759   ║
║  Confidence gap (TP - FP)                                    +0.0876   ║
╠════════════════════════════════════════════════════════════════════════╣
║  2. Boundary Error Anal

In [17]:
# CELL 14 — EXAMPLES REPORT
# Top 3 + Bottom 3 for each metric, with full token-level display
# ════════════════════════════════════════════════════════════════
 
def to_bioes_tags(raw: str, spans: list, label_key: str = "entity_group") -> list:
    tokens  = list(re.finditer(r"\S+", raw))
    labels  = ["O"] * len(tokens)
    for span in sorted(spans, key=lambda s: s["start"]):
        in_idx = [wi for wi, wm in enumerate(tokens)
                  if wm.start() < span["end"] and wm.end() > span["start"]]
        if not in_idx: continue
        lbl = span[label_key]
        if len(in_idx) == 1:
            labels[in_idx[0]] = f"S-{lbl}"
        else:
            labels[in_idx[0]]  = f"B-{lbl}"
            labels[in_idx[-1]] = f"E-{lbl}"
            for idx in in_idx[1:-1]:
                labels[idx] = f"I-{lbl}"
    return [(tokens[i].group(), labels[i]) for i in range(len(tokens))]
 
def print_example(rank_label: str, row_: pd.Series, idx: int):
    raw_     = row_["rawText"]
    gt_mask_ = row_["maskedText"]
    norm_mask= row_["normalized_masked_text"]
    raw_mask = row_["raw_masked_text"]
 
    gt_s_      = json.loads(row_["gt_spans"])
    raw_s_     = json.loads(row_["raw_predicted_spans"])     # ✅ true raw spans
    norm_s_    = json.loads(row_["normalized_predicted_spans"])
 
    # ✅ FIX: raw BIOES uses raw_s_ (true raw model output), norm BIOES uses norm_s_
    bioes_gt   = to_bioes_tags(raw_, gt_s_,  label_key="label")
    bioes_raw  = to_bioes_tags(raw_, raw_s_, label_key="entity_group")
    bioes_norm = to_bioes_tags(raw_, norm_s_,label_key="entity_group")
 
    print(f"\n  {'━'*84}")
    print(f"  {rank_label}  |  Row s_no={int(row_['s_no'])}")
    print(f"  category={row_['category']}  difficulty={row_['difficulty']}  adversarial={row_['adversarial_type']}")
    print(f"  {'━'*84}")
 
    # Scores block
    print(f"  {'Metric':<28} {'Precision':>10} {'Recall':>10} {'F1':>10}")
    print(f"  {'─'*62}")
    print(f"  {'Raw Span':<28} {row_['raw_span_precision']:>10.4f} {row_['raw_span_recall']:>10.4f} {row_['raw_span_f1']:>10.4f}")
    print(f"  {'Norm Span':<28} {row_['norm_span_precision']:>10.4f} {row_['norm_span_recall']:>10.4f} {row_['norm_span_f1']:>10.4f}")
    print(f"  {'Token':<28} {row_['token_precision']:>10.4f} {row_['token_recall']:>10.4f} {row_['token_f1']:>10.4f}")
    print(f"  {'Agnostic (any label)':<28} {row_['agnostic_precision']:>10.4f} {row_['agnostic_recall']:>10.4f} {row_['agnostic_f1']:>10.4f}")
 
    print(f"\n  INPUT     : {raw_}")
    print(f"  GT MASKED : {gt_mask_}")
    print(f"  RAW PRED  : {raw_mask}")
    print(f"  NORM PRED : {norm_mask}")
    if row_["missed_categories"]:
        print(f"  ❌ MISSED : {row_['missed_categories']}")
    if row_["hallucinated_categories"]:
        print(f"  ⚠️  HALLUC : {row_['hallucinated_categories']}")
 
    # BIOES token table — 4 columns: token | GT | Raw | Norm | match indicators
    print(f"\n  {'Token':<28} {'GT Tag':<24} {'Raw Tag':<24} {'Norm Tag':<24}  Span  Agn")
    print(f"  {'─'*110}")
    for i, ((tok, gt_tag), (_, raw_tag), (_, norm_tag)) in enumerate(
            zip(bioes_gt, bioes_raw, bioes_norm)):
        gt_d   = gt_tag   if gt_tag   != "O" else "·"
        raw_d  = raw_tag  if raw_tag  != "O" else "·"
        norm_d = norm_tag if norm_tag != "O" else "·"
        # span match: norm tag == gt tag (exact label + BIOES position)
        span_ok = "✓" if norm_tag == gt_tag else ("✗" if gt_tag != "O" or norm_tag != "O" else " ")
        # agnostic match: both are non-O (PII was tagged, any label)
        agn_ok  = "✓" if (gt_tag != "O" and norm_tag != "O") else \
                  ("✗" if gt_tag != "O" and norm_tag == "O" else \
                  ("!" if gt_tag == "O" and norm_tag != "O" else " "))
        print(f"  {tok:<28} {gt_d:<24} {raw_d:<24} {norm_d:<24}  {span_ok:<5} {agn_ok}")
 
METRICS_TO_SHOW = [
    ("norm_span_f1",  "Normalized Span F1"),
    ("token_f1",      "Token F1"),
    ("raw_span_f1",   "Raw Span F1"),
]
N_EXAMPLES = 3
 
print("\n" + "╔" + "═"*78 + "╗")
print(f"║{'  EXAMPLES REPORT — Highest & Lowest Scoring Rows':^78}║")
print("╚" + "═"*78 + "╝")
 
for metric_col, metric_name in METRICS_TO_SHOW:
    print(f"\n\n{'═'*82}")
    print(f"  METRIC: {metric_name}  [{metric_col}]")
    print(f"{'═'*82}")
 
    # filter rows that actually have GT spans (skip empty GT rows for fairness)
    valid = df_out[df_out["gt_span_count"] > 0].copy()
    sorted_asc  = valid.sort_values(metric_col, ascending=True)
    sorted_desc = valid.sort_values(metric_col, ascending=False)
 
    print(f"\n  ── LOWEST {N_EXAMPLES}  (worst performance) ──")
    for rank, (_, row_) in enumerate(sorted_asc.head(N_EXAMPLES).iterrows(), 1):
        print_example(f"LOWEST #{rank}", row_, rank)
 
    print(f"\n  ── HIGHEST {N_EXAMPLES}  (best performance) ──")
    for rank, (_, row_) in enumerate(sorted_desc.head(N_EXAMPLES).iterrows(), 1):
        print_example(f"HIGHEST #{rank}", row_, rank)
 
print(f"\n\n✅ Examples report complete.")


╔══════════════════════════════════════════════════════════════════════════════╗
║                EXAMPLES REPORT — Highest & Lowest Scoring Rows               ║
╚══════════════════════════════════════════════════════════════════════════════╝


══════════════════════════════════════════════════════════════════════════════════
  METRIC: Normalized Span F1  [norm_span_f1]
══════════════════════════════════════════════════════════════════════════════════

  ── LOWEST 3  (worst performance) ──

  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  LOWEST #1  |  Row s_no=685
  category=private_address  difficulty=hard  adversarial=slash_in_address_vs_code
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Metric                        Precision     Recall         F1
  ──────────────────────────────────────────────────────────────
  Raw Span                         0.0000     0.0000     0.0000
  Norm Span              